Cell 1: Import Libraries and Generate Dataset

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np

In [2]:
# Set random seed for reproducibility
np.random.seed(42)

In [3]:
# Define the number of samples
num_samples = 100

In [4]:
# Generate synthetic data
data = {
    'area': np.random.randint(500, 2000, size=num_samples),  # Area in square feet
    'number_of_bedrooms': np.random.randint(1, 5, size=num_samples),  # Number of bedrooms (1 to 4)
    'age_of_property': np.random.randint(0, 30, size=num_samples),  # Age of property in years
    'location_rating': np.random.uniform(1, 10, size=num_samples)  # Location rating (1 to 10)
}

In [5]:
# Create a DataFrame
df = pd.DataFrame(data)

In [6]:
# Define a simple linear relationship for flat prices
df['flat_price'] = (df['area'] * 100) + (df['number_of_bedrooms'] * 50000) - (df['age_of_property'] * 2000) + (df['location_rating'] * 10000)

In [7]:
# Display the first few rows of the DataFrame
df.head()

,area,number_of_bedrooms,age_of_property,location_rating,flat_price
0,1626,4,5,6.318036,415780.364887
1,1959,1,27,7.098079,262880.792566
2,1360,1,27,1.149290,143492.904604
3,1794,1,11,5.608838,263488.375247
4,1630,3,11,3.038462,321384.619768


Cell 2: Decision Tree Regression Using Scikit-Learn

In [8]:
# Import necessary libraries for scikit-learn decision tree regression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

In [9]:
# Select features and target variable for scikit-learn model
X_sklearn = df[['area', 'number_of_bedrooms', 'age_of_property', 'location_rating']]
y_sklearn = df['flat_price']

In [10]:
# Split the dataset into training and testing sets (80% train, 20% test)
X_train_sklearn, X_test_sklearn, y_train_sklearn, y_test_sklearn = train_test_split(X_sklearn, y_sklearn, test_size=0.2, random_state=42)

In [11]:
# Create and fit a Decision Tree Regressor model using scikit-learn
model_sklearn = DecisionTreeRegressor(random_state=42)
model_sklearn.fit(X_train_sklearn, y_train_sklearn)

DecisionTreeRegressor(random_state=42)

In [12]:
# Make predictions on the test set using scikit-learn model
y_pred_sklearn = model_sklearn.predict(X_test_sklearn)

In [13]:
# Evaluate the scikit-learn model using Mean Squared Error (MSE)
mse_sklearn = mean_squared_error(y_test_sklearn, y_pred_sklearn)
print("Scikit-learn Decision Tree Mean Squared Error:", mse_sklearn)

Scikit-learn Decision Tree Mean Squared Error: 1348469120.5455093


In [15]:
# Predicting a price for a specific house using scikit-learn model
sample_house = np.array([[1500, 3, 10, 8]]) 

In [16]:
# Example: area=1500 sq ft, bedrooms=3, age=10 years, location rating=8
predicted_price_sklearn = model_sklearn.predict(sample_house)
print("Predicted price using Scikit-learn Decision Tree:", predicted_price_sklearn[0])

Predicted price using Scikit-learn Decision Tree: 331069.87696743716


C:\Users\vaish\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeRegressor was fitted with feature names
  warnings.warn(


Cell 3: Manual Decision Tree Regression Implementation

In [22]:
# Define a simple decision tree regressor class
class SimpleDecisionTreeRegressor:
    def __init__(self):
        self.tree = None

    def fit(self, X, y):
        self.tree = self._build_tree(X, y)

    def _build_tree(self, X, y):
        if len(set(y)) == 1:  # If all values are the same
            return {'value': y[0]}
        
        if len(X) == 0:
            return {'value': np.mean(y)}

        best_feature_index = self._best_split(X, y)
        if best_feature_index is None:
            return {'value': np.mean(y)}

        tree = {'feature_index': best_feature_index}
        left_indices = X[:, best_feature_index] < np.median(X[:, best_feature_index])
        right_indices = ~left_indices
        
        tree['left'] = self._build_tree(X[left_indices], y[left_indices])
        tree['right'] = self._build_tree(X[right_indices], y[right_indices])
        
        return tree

    def _best_split(self, X, y):
        min_mse = float('inf')
        best_index = None
        
        for feature_index in range(X.shape[1]):
            left_indices = X[:, feature_index] < np.median(X[:, feature_index])
            right_indices = ~left_indices
            
            if len(y[left_indices]) == 0 or len(y[right_indices]) == 0:
                continue
            
            mse_left = np.mean((y[left_indices] - np.mean(y[left_indices])) ** 2)
            mse_right = np.mean((y[right_indices] - np.mean(y[right_indices])) ** 2)
            mse_total = (len(y[left_indices]) * mse_left + len(y[right_indices]) * mse_right) / len(y)

            if mse_total < min_mse:
                min_mse = mse_total
                best_index = feature_index
                
        return best_index

    def predict(self, X):
        return np.array([self._predict_single(x) for x in X])

    def _predict_single(self, x):
        node = self.tree
        while 'feature_index' in node:
            feature_value = x[node['feature_index']]
            if feature_value < np.median(x[node['feature_index']]):
                node = node['left']
            else:
                node = node['right']
        return node['value']

# Prepare data for manual regression
X_manual = df[['area', 'number_of_bedrooms', 'age_of_property', 'location_rating']].values
y_manual = df['flat_price'].values

# Create and fit a Simple Decision Tree Regressor model manually
manual_model = SimpleDecisionTreeRegressor()
manual_model.fit(X_manual, y_manual)

# Make predictions using the manual model on test set
y_pred_manual = manual_model.predict(X_test_sklearn.values)

In [23]:
# Evaluate the manual model using Mean Squared Error (MSE)
mse_manual = mean_squared_error(y_test_sklearn.values, y_pred_manual)
print("Manual Decision Tree Mean Squared Error:", mse_manual)

Manual Decision Tree Mean Squared Error: 27209815694.335594


In [24]:
# Predicting a price for a specific house using manual model
predicted_price_manual = manual_model.predict(sample_house)
print("Predicted price using Manual Decision Tree:", predicted_price_manual[0])

Predicted price using Manual Decision Tree: 429142.6817007476


Cell 4: Comparison of Both Methods

In [26]:
# Compare both methods' performance
print("\nComparison of Scikit-learn vs Manual Decision Tree Regression:")
print(f"Scikit-learn MSE: {mse_sklearn:.2f}")
print(f"Manual MSE: {mse_manual:.2f}")

if mse_manual < mse_sklearn:
    print("The manual implementation performed better.")
elif mse_manual > mse_sklearn:
    print("The scikit-learn implementation performed better.")
else:
    print("Both implementations performed equally well.")


Comparison of Scikit-learn vs Manual Decision Tree Regression:
Scikit-learn MSE: 1348469120.55
Manual MSE: 27209815694.34
The scikit-learn implementation performed better.
